# Polyomino decision classifier on TPU v5e-8

Run dense training or start QAT from an existing **full training-state checkpoint** on Kaggle or Colab. Choose a runtime exposing 8 local v5e TPU devices. The notebook kernel never imports JAX; pinned Python 3.12 child processes own the TPU. The device probe stops if the runtime exposes fewer than 8 devices.

Set `TRAINING_MODE = "qat"`, choose `QAT_QUANTIZATION` (`"nf4"` or `"ternary"`), and point `DENSE_CHECKPOINT_DIR` to the directory containing the old `manifest.json` and its `state.safetensors` or `model.safetensors`. On Kaggle, attach that checkpoint as a read-only dataset under `/kaggle/input`. QAT loads the trained FP32 parameters, including the action head and frozen embeddings, then starts fresh Adam state and a new data cursor. Projection matrices use fake quantization; embeddings and the action head stay dense. The 2-update smoke verifies every saved state tensor before gameplay.

Kaggle scratch checkout, venv, model metadata, and dataset cache live under `/kaggle/temp`; saved long-run checkpoints belong under `/kaggle/working`. Save or download `/kaggle/working` outputs before the session ends. Colab uses `/content`; mount persistent storage for a long run. Both hosts need internet access for the pinned repository, model metadata, dataset, and Python packages. No TPU QAT throughput, memory, or model-quality result has been measured yet.


In [ ]:
import hashlib
import json
import math
from pathlib import Path
import subprocess
import sys

IS_KAGGLE = Path("/kaggle/working").is_dir()
HOST_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path("/content")
SCRATCH = Path("/kaggle/temp") if IS_KAGGLE else HOST_ROOT
SCRATCH.mkdir(parents=True, exist_ok=True)
CHECKOUT = SCRATCH / "minifield-training-v5e8"
ROOT = SCRATCH / "polyomino-tpu-v5e8"
VENV = ROOT / "venv"
PYTHON = VENV / "bin/python"
SOURCE_REVISION = "79d9bcfc302b743efbb439d151385e08236c33b4"
SOURCE_BUNDLE = HOST_ROOT / "minifield-training-tpu-v5e-8.bundle"
SOURCE_REPO_URL = "https://github.com/Minifield-Labs/minifield-training.git"
BASE_REVISION = "9d2be5519834990d30996f878b6771cccbd24f2c"
TRAINING_MODE = "dense"  # Set "qat" to start from DENSE_CHECKPOINT_DIR.
QAT_QUANTIZATION = "nf4"  # Or "ternary"; used only in QAT mode.
DENSE_CHECKPOINT_DIR: Path | None = None  # Example: Path("/kaggle/input/model100k/Model100k")
ROOT.mkdir(parents=True, exist_ok=True)

def run_child(*command: str, cwd: Path | None = None) -> str:
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        assert process.stdout is not None
        lines = []
        for line in process.stdout:
            print(line, end="", flush=True)
            lines.append(line)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    return "".join(lines)

UV = ROOT / "tooling/bin/uv"
if not UV.is_file():
    run_child(sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
              "--target", str(ROOT / "tooling"), "uv==0.11.30")
assert "uv 0.11.30" in run_child(str(UV), "--version")


In [ ]:
if not (CHECKOUT / ".git").is_dir():
    assert not CHECKOUT.exists(), f"Expected a clean path: {CHECKOUT}"
    clone_source = str(SOURCE_BUNDLE) if SOURCE_BUNDLE.is_file() else SOURCE_REPO_URL
    run_child("git", "clone", clone_source, str(CHECKOUT))
assert not run_child("git", "status", "--porcelain", cwd=CHECKOUT).strip()
contains_pin = subprocess.run(["git", "cat-file", "-e", f"{SOURCE_REVISION}^{{commit}}"], cwd=CHECKOUT).returncode == 0
if not contains_pin:
    run_child("git", "fetch", SOURCE_REPO_URL, "feat/quantization-strategies", cwd=CHECKOUT)
    assert subprocess.run(["git", "cat-file", "-e", f"{SOURCE_REVISION}^{{commit}}"], cwd=CHECKOUT).returncode == 0
run_child("git", "checkout", "--detach", SOURCE_REVISION, cwd=CHECKOUT)
assert run_child("git", "rev-parse", "HEAD", cwd=CHECKOUT).strip() == SOURCE_REVISION
assert (CHECKOUT / "examples/polyomino/train.py").is_file()


In [ ]:
if not PYTHON.is_file():
    # uv provisions Python 3.12 when the notebook host doesn't provide it.
    run_child(str(UV), "venv", "--python", "3.12", str(VENV))
run_child(str(UV), "pip", "install", "--python", str(PYTHON),
          "jax[tpu]==0.7.2", "tensorflow-cpu==2.20.0")
run_child(str(UV), "pip", "install", "--python", str(PYTHON),
          ".[numerical,storage,text,hub]", cwd=CHECKOUT)
probe = """
import jax
print('JAX:', jax.__version__, flush=True)
print('Devices:', [(d.id, d.platform, d.device_kind) for d in jax.devices()], flush=True)
assert jax.__version__ == '0.7.2'
assert jax.process_count() == 1, 'This notebook requires a single host'
assert jax.device_count() == jax.local_device_count() == 8, 'Select a runtime exposing 8 TPU devices'
assert all(d.platform == 'tpu' for d in jax.devices()), 'TPU runtime required'
"""
run_child(str(PYTHON), "-c", probe)
train_help = run_child(str(PYTHON), "-m", "examples.polyomino.train", "--help", cwd=CHECKOUT)
assert all(flag in train_help for flag in ("--devices", "--quantization", "--warm-start-source-id", "--warm-start-tensor-file"))


In [ ]:
MODEL_DIR = ROOT / "base-model"
DATASET_CACHE = ROOT / "dataset-cache"
assert TRAINING_MODE in {"dense", "qat"}
assert QAT_QUANTIZATION in {"nf4", "ternary"}
run_child(str(UV), "pip", "install", "--python", str(PYTHON), "huggingface_hub")
patterns = ["config.json", "tokenizer.json"]
if TRAINING_MODE == "dense":
    patterns.append("model.safetensors")
download = "from huggingface_hub import snapshot_download; import json, sys; snapshot_download(repo_id='LiquidAI/LFM2.5-230M-Base', revision=sys.argv[1], allow_patterns=json.loads(sys.argv[2]), local_dir=sys.argv[3])"
run_child(str(PYTHON), "-c", download, BASE_REVISION, json.dumps(patterns), str(MODEL_DIR), cwd=CHECKOUT)
assert all((MODEL_DIR / name).is_file() for name in patterns)
print("Pinned Base files ready. The dataset downloads on the first training call.")


The pinned dataset contains 4,737,585 decisions. Its first load downloads about 2.5 GB of Parquet plus a disk-backed Arrow cache. A full FP32 checkpoint is about 2.75 GB. QAT still saves full FP32 masters and Adam moments, so allow the same checkpoint space. The optional effective-weight export writes dense FP32 weights; it isn't a packed model bundle.

`--rows` is the global physical batch size. The 8-device recipe uses 16 rows per microbatch, 4 microbatches per update, and 512 tokens. The QAT learning rate below is a configurable starting value, not a measured recommendation. Source, quantizer, batch settings, and model lineage are bound into the saved recipe.


In [ ]:
DEVICE_COUNT = 8
ROWS_PER_DEVICE = 2
MICROBATCHES = 4
SEQUENCE_LENGTH = 512
DENSE_LEARNING_RATE = 0.0001
QAT_LEARNING_RATE = 0.00001  # Adjust only with a fresh QAT run directory.
LEARNING_RATE = QAT_LEARNING_RATE if TRAINING_MODE == "qat" else DENSE_LEARNING_RATE
QUANTIZATION = QAT_QUANTIZATION if TRAINING_MODE == "qat" else "dense"
GLOBAL_ROWS = DEVICE_COUNT * ROWS_PER_DEVICE
DATASET_ID = "protodotdesign/polyomino-decisions-v1"
DATASET_REVISION = "d1a79caa4eaeba129630f858f9c2de7d6de7533a"
DATASET_ROWS = 4_737_585
TOTAL_UPDATES = math.ceil(DATASET_ROWS / (MICROBATCHES * GLOBAL_ROWS))
recipe_args = ["--devices", str(DEVICE_COUNT), "--rows", str(GLOBAL_ROWS),
               "--microbatches", str(MICROBATCHES), "--sequence-length", str(SEQUENCE_LENGTH),
               "--learning-rate", str(LEARNING_RATE), "--quantization", QUANTIZATION]

DENSE_CURSOR = None
DENSE_TENSOR_FILE = None
DENSE_MANIFEST_SHA256 = None
DENSE_TENSOR_SHA256 = None
if TRAINING_MODE == "qat":
    assert DENSE_CHECKPOINT_DIR is not None, "Set DENSE_CHECKPOINT_DIR to an attached full checkpoint"
    assert DENSE_CHECKPOINT_DIR.is_absolute() and DENSE_CHECKPOINT_DIR.is_dir()
    manifest_path = DENSE_CHECKPOINT_DIR / "manifest.json"
    manifest_bytes = manifest_path.read_bytes()
    dense_manifest = json.loads(manifest_bytes)
    assert dense_manifest["format"] == "minifield.full-training-state/1"
    DENSE_CURSOR = dense_manifest["cursor"]
    assert all(DENSE_CURSOR[key] for key in ("run_id", "source_id", "data_sha256"))
    assert isinstance(DENSE_CURSOR["next_batch"], int)
    DENSE_TENSOR_SHA256 = dense_manifest["tensor_sha256"]
    assert isinstance(DENSE_TENSOR_SHA256, str) and len(DENSE_TENSOR_SHA256) == 64
    tensor_candidates = [name for name in ("state.safetensors", "model.safetensors")
                         if (DENSE_CHECKPOINT_DIR / name).is_file()]
    assert len(tensor_candidates) == 1, "Attach exactly one full-state tensor file"
    DENSE_TENSOR_FILE = tensor_candidates[0]
    DENSE_MANIFEST_SHA256 = hashlib.sha256(manifest_bytes).hexdigest()
    print(f"QAT source: {DENSE_CHECKPOINT_DIR}, dense step {DENSE_CURSOR['next_batch']}, tensor {DENSE_TENSOR_FILE}")

def start_mode_args(start: str) -> list[str]:
    assert start in {"warm", "resume"}
    if start == "resume":
        return ["--resume-latest"]
    if TRAINING_MODE == "dense":
        return []
    assert DENSE_CHECKPOINT_DIR is not None and DENSE_CURSOR is not None and DENSE_TENSOR_FILE is not None
    return ["--warm-start-checkpoint", str(DENSE_CHECKPOINT_DIR),
            "--warm-start-run-id", DENSE_CURSOR["run_id"],
            "--warm-start-source-id", DENSE_CURSOR["source_id"],
            "--warm-start-tensor-file", DENSE_TENSOR_FILE]

recipe_signature = json.dumps({
    "source_revision": SOURCE_REVISION,
    "base_revision": BASE_REVISION,
    "training_mode": TRAINING_MODE,
    "quantization": QUANTIZATION,
    "learning_rate": LEARNING_RATE,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "dataset_rows": DATASET_ROWS,
    "args": recipe_args,
    "dense_manifest_sha256": DENSE_MANIFEST_SHA256,
    "dense_tensor_sha256": DENSE_TENSOR_SHA256,
    "dense_data_identity": DENSE_CURSOR["data_sha256"] if DENSE_CURSOR else None,
}, sort_keys=True)
print(f"{GLOBAL_ROWS} global rows, {ROWS_PER_DEVICE} rows/device, {GLOBAL_ROWS * MICROBATCHES} decisions/update")
print(f"One dataset pass: {TOTAL_UPDATES} updates, including the padded final update")


The smoke runs 2 updates and reloads its checkpoint to compare every parameter, optimizer moment, step, and cursor. QAT starts from the attached dense checkpoint with new Adam moments and a new cursor. Gameplay then uses the saved policy with the selected quantization plan. If a smoke was interrupted after writing an unverified checkpoint, choose a fresh scratch `ROOT` and rerun setup.


In [ ]:
CHECKPOINTS = ROOT / f"checkpoints-{TRAINING_MODE}-{QUANTIZATION}-roundtrip"
smoke_run_id = f"polyomino-v5e8-{TRAINING_MODE}-{QUANTIZATION}-smoke-2"
smoke_checkpoint = CHECKPOINTS / "step-00000002" / "manifest.json"
smoke_verified = CHECKPOINTS / "roundtrip-verified"
if not smoke_verified.is_file():
    assert not list(CHECKPOINTS.glob("step-*/manifest.json")), "Use a fresh ROOT for an interrupted, unverified smoke"
    smoke_output = run_child(str(PYTHON), "-u", "-m", "examples.polyomino.train",
              "--model-dir", str(MODEL_DIR), "--dataset-cache", str(DATASET_CACHE),
              "--checkpoint-root", str(CHECKPOINTS), "--run-id", smoke_run_id,
              "--platform", "tpu", *recipe_args, *start_mode_args("warm"), "--max-steps", "2",
              "--checkpoint-every", "2", "--report-every", "1",
              "--verify-checkpoint", "--eval-games", "0", cwd=CHECKOUT)
    assert '"checkpoint_roundtrip_verified":1.0' in smoke_output
    assert smoke_checkpoint.is_file()
    smoke_verified.write_text(recipe_signature + "\n")
assert smoke_checkpoint.is_file()
assert smoke_verified.read_text().strip() == recipe_signature
run_child(str(PYTHON), "-u", "-m", "examples.polyomino.evaluate",
          "--model-dir", str(MODEL_DIR), "--checkpoint", str(smoke_checkpoint.parent),
          "--run-id", smoke_run_id, "--replay-dir", str(CHECKPOINTS / "replays"),
          *recipe_args, "--games", "1", "--max-ticks", "1000", cwd=CHECKOUT)


Set `PERSISTENT_ROOT` to a writable output directory to start the optional long run. On Kaggle, use `/kaggle/working`; save or download its outputs before the session ends. On Colab, use mounted storage. The mode-specific recipe file prevents accidental reuse after changing the source checkpoint, quantizer, learning rate, or batch settings.

The first QAT invocation warm starts from the attached dense checkpoint. Every later invocation resumes the new QAT checkpoint at its next unread update, with its own strict 8-device identity. Each call runs for up to 3 hours or the remaining dataset updates. Checkpoints are written every 5,000 updates and at a normal stop; a disconnected runtime can lose work since the last saved checkpoint.


In [ ]:
PERSISTENT_ROOT: Path | None = None  # Kaggle: Path("/kaggle/working"); Colab: mounted storage.
if PERSISTENT_ROOT is None:
    print("Set PERSISTENT_ROOT to run training.")
else:
    assert PERSISTENT_ROOT.is_absolute() and PERSISTENT_ROOT.is_dir()
    assert not PERSISTENT_ROOT.is_relative_to(ROOT), "Keep long-run outputs outside scratch"
    long_checkpoints = PERSISTENT_ROOT / f"polyomino-v5e8-{TRAINING_MODE}-{QUANTIZATION}-checkpoints"
    long_checkpoints.mkdir(parents=True, exist_ok=True)
    recipe_file = long_checkpoints / "recipe.json"
    if recipe_file.exists():
        assert recipe_file.read_text().strip() == recipe_signature, "Use a fresh directory for a changed recipe"
    else:
        assert not list(long_checkpoints.glob("step-*/manifest.json")), "Existing checkpoints lack a verified recipe"
        recipe_file.write_text(recipe_signature + "\n")
    run_id = f"polyomino-{TRAINING_MODE}-{QUANTIZATION}-decisions-v1-v5e8"
    checkpoints = sorted(long_checkpoints.glob("step-*/manifest.json"))
    completed = json.loads(checkpoints[-1].read_text())["cursor"]["next_batch"] if checkpoints else 0
    assert isinstance(completed, int) and 0 <= completed <= TOTAL_UPDATES
    remaining = TOTAL_UPDATES - completed
    if remaining:
        start = "resume" if checkpoints else "warm"
        run_child(str(PYTHON), "-u", "-m", "examples.polyomino.train",
                  "--model-dir", str(MODEL_DIR), "--dataset-cache", str(DATASET_CACHE),
                  "--checkpoint-root", str(long_checkpoints), "--run-id", run_id,
                  "--platform", "tpu", *recipe_args, *start_mode_args(start),
                  "--max-hours", "3", "--max-steps", str(remaining),
                  "--checkpoint-every", "5000", "--report-every", "50",
                  "--eval-games", "0", cwd=CHECKOUT)
    else:
        print("All decisions have already been visited.")
    checkpoints = sorted(long_checkpoints.glob("step-*/manifest.json"))
    assert checkpoints, "Training produced no checkpoint"
    latest = checkpoints[-1].parent
    run_child(str(PYTHON), "-u", "-m", "examples.polyomino.evaluate",
              "--model-dir", str(MODEL_DIR), "--checkpoint", str(latest),
              "--run-id", run_id, "--replay-dir", str(long_checkpoints / "replays"),
              *recipe_args, "--games", "3", "--max-ticks", "2000", cwd=CHECKOUT)
